# 第 13 章　分析手順と再現性

「前年と比べて，何時台の貸出が増えたか」を調べると，「登録区分でも同じか」「別の月ではどうか」という問いが続きます。そのたびに集計方法まで変わってしまうと，結果の違いが何によるものか分かりにくくなります。

このノートでは，同じ集計を関数として用意し，表や列だけを替えて比較します。最後に最初から再実行し，報告した値を元の記録から確かめられる状態にします。実習では登録・未登録の一方を選び，6 月と 12 月の時間帯別の利用を説明します。

前半は，章本文の説明に合わせて，上から順にセルを実行します。コードの前に何を調べるかを読み，実行後は表や図のどの値が問いへの答えになるかを確かめてください。後半の約 20 分の実習では，比べる対象や条件を自分で選び，結果を短い文章にします。前半のコードは実行できる見本なので，作図の処理をすべて書き写す必要はありません。

Colab では「ドライブにコピーを保存」で自分用のノートを作ります。CSV は，手元にファイルがなければ公開 URL から読み込みます。ローカルでは，配布 ZIP の配置のまま開くか，ノートと CSV を同じフォルダーに置いて実行できます。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly["date"] = pd.to_datetime(hourly["dteday"])
print("Python:", sys.version.split()[0], "pandas:", pd.__version__)
hourly[["date", "hr", "cnt", "registered", "casual"]].head(3)

## 本文と一緒に実行

### 1. 二つの年の 6 月を比べる

2011 年と 2012 年の 6 月で，朝と夕方の平均貸出件数はどれくらい違うでしょうか。まず対象期間の行を選び，同じ時間帯の件数を平均して図にします。

図を描いた後は，2012 年の 8 時台に注目します。一つの平均点が何日分の記録から作られたかを，行数・合計・平均で確かめてください。次に，すべての時間帯で同じ三つの値を求めます。この計算を，後で別の入力にも使える関数にします。

In [ ]:
june_2011 = hourly.loc[
    (hourly["date"].dt.year == 2011) & (hourly["date"].dt.month == 6)
]
june_2012 = hourly.loc[
    (hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 6)
]
print(len(june_2011), len(june_2012))

In [ ]:
plt.rcParams["figure.figsize"] = (6.6, 3.7)
fig, ax = plt.subplots(layout="constrained")
for records, label, style in [
    (june_2011, "June 2011", "o-"), (june_2012, "June 2012", "s--")
]:
    means = records.groupby("hr")["cnt"].mean()
    ax.plot(means.index, means, style, markersize=3, label=label)
ax.set(xlabel="Clock hour", ylabel="Mean rental count", xticks=range(0, 24, 3), ylim=(0, 750))
ax.legend(frameon=False)
plt.show()

In [ ]:
eight_in_june = june_2012.loc[june_2012["hr"] == 8, "cnt"]
print("Recorded hours:", len(eight_in_june))
print("Total rentals:", eight_in_june.sum())
print("Mean rentals:", eight_in_june.mean())

In [ ]:
grouped = june_2012.groupby("hr")["cnt"]
june_summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
june_summary.loc[[8, 13, 17]].round(2)

2012 年 6 月の 8 時台は，30 行の貸出合計が 15,252 件，平均が 508.4 件です。30 は貸出件数ではなく，異なる 30 日の 8 時台を表す行数です。

この計算で替えたいのは，対象となる表と，平均する列です。時間帯で分けて行数・合計・平均を求める手順は共通にできます。次に，表と列名を受け取る関数を定義します。

### 2. 同じ集計を使える関数にする

二つの年を同じ方法で比べるため，集計手順を `hourly_summary()` にまとめます。定義セルを実行しただけでは，特定の月の集計はまだ行われません。実際の表と列名を引数として渡したときに，処理が動きます。

まず 2012 年，次に 2011 年の 6 月を渡します。呼び出しに渡す表と列名の文字列が，関数内の `records` と `value_column` にどのように対応するかを読んでください。返された二つの表を同じ時間帯どうしで並べ，平均の差と増加率を計算します。

In [ ]:
def hourly_summary(records, value_column):
    grouped = records.groupby("hr")[value_column]
    summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
    return summary

In [ ]:
profile_2012 = hourly_summary(june_2012, "cnt")
profile_2012.loc[[8, 13, 17]].round(2)

In [ ]:
profile_2011 = hourly_summary(june_2011, "cnt")

comparison = pd.DataFrame({
    "June 2011": profile_2011["mean"],
    "June 2012": profile_2012["mean"],
})
comparison["increase"] = comparison["June 2012"] - comparison["June 2011"]
comparison.loc[[8, 13, 17]].round(2)

In [ ]:
comparison["increase_pct"] = 100 * comparison["increase"] / comparison["June 2011"]
comparison.loc[[8, 17], ["increase", "increase_pct"]].round(2)

返り値は，各時間帯を一行にまとめた 24 行の表です。8 時台と 17 時台について，増加件数が大きいのはどちらか，増加率が大きいのはどちらかを比べてください。増加率は前年の平均を分母にするので，増加件数と同じ順になるとは限りません。

関数を共通にしたことで，どちらの年にも同じ集計を使えました。ただし，件数と増加率のどちらを報告するかは，担当者が知りたいことに応じて選びます。

### 3. 表示と返り値の役割を確かめる

年ごとの平均を引き算するには，関数で作った表を後の計算でも使える必要があります。次の `show_eight()` は値を画面に表示しますが，集計表を返す行がありません。呼び出した後の `shown` に何が入るかを，実行前に予想してください。

続くセルでは，元の入力表と，先ほど受け取った集計表の行数を比べます。集計によって元の記録が 24 行へ書き換わったのか，別の表が作られたのかも確かめます。

In [ ]:
def show_eight(records, value_column):
    table = hourly_summary(records, value_column)
    print(table.loc[8])


shown = show_eight(june_2012, "cnt")
print("Returned None:", shown is None)

In [ ]:
print("Input rows:", len(june_2012))
print("Returned rows:", len(profile_2012))

値は画面に表示されましたが，`shown` は `None` です。一方，`hourly_summary()` は表を返すので，受け取った表から差を計算したり図を描いたりできます。画面に見えることと，後の計算へ値を渡せることは別です。

入力の `june_2012` は 720 行のままで，返された集計表は 24 行です。元の記録が残っているので，図の気になる時間帯を一日ずつ調べ直せます。

### 4. 同じ期間で利用区分を替える

全体では朝と夕方に利用が多くなりました。登録区分だけを数えても，同じ時間帯に利用が多いのでしょうか。対象は両年の 6 月のまま，集計列を登録区分へ替えます。

呼び出しにパラメーター名を添えると，どの値を渡したかが読みやすくなります。表を渡す指定と列名を渡す指定を見分け，全体と登録区分の図を比べてください。

In [ ]:
registered_2011 = hourly_summary(records=june_2011, value_column="registered")
registered_2012 = hourly_summary(records=june_2012, value_column="registered")

registered_comparison = pd.DataFrame({
    "June 2011": registered_2011["mean"],
    "June 2012": registered_2012["mean"],
})
registered_comparison.loc[[8, 13, 17]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (7.0, 3.7)
fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, layout="constrained")
for ax, title, first, second in [
    (axes[0], "All rentals", profile_2011, profile_2012),
    (axes[1], "Registered", registered_2011, registered_2012),
]:
    ax.plot(first.index, first["mean"], "o-", markersize=3, label="June 2011")
    ax.plot(second.index, second["mean"], "s--", markersize=3, label="June 2012")
    ax.set(title=title, xlabel="Clock hour", xticks=[0, 6, 12, 18, 23], ylim=(0, 750))
axes[0].set_ylabel("Mean rental count")
axes[1].legend(frameon=False)
plt.show()

2012 年 6 月の 8 時台では，登録区分の平均が 472.6 件です。全体の 508.4 件との差は，未登録区分の平均に当たります。図の朝夕の件数のうち，登録区分がどれくらいを占めるかを調べられます。

ただし，登録していることは通勤目的で使ったことを意味しません。本文と同じく，利用区分の内訳として分かったことと，移動の目的についてはまだ分からないことを分けます。

### 5. 月を替えて平均の分母も確かめる

6 月に見えた朝夕の利用は，12 月でも同じでしょうか。今度は全体件数の列を使い，入力表だけを 2012 年 12 月へ替えます。8 時台と 17 時台の平均の差が，6 月とどう違うかを図で確かめます。

同じ関数が動いても，各時刻に何日分の記録があるかは同じとは限りません。12 月の行数を調べ，平均に使った行が月の日数より少ない時間帯を確認してください。最後の照合セルでは，6 月の集計結果を元の行数・合計・平均と比べ，意図した記録が計算に含まれたかを確かめます。

In [ ]:
december_2012 = hourly.loc[
    (hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 12)
]
profile_december = hourly_summary(december_2012, "cnt")
profile_december.loc[[3, 4, 8, 13, 17]].round(2)

In [ ]:
plt.rcParams["figure.figsize"] = (6.6, 3.7)
fig, ax = plt.subplots(layout="constrained")
ax.plot(profile_2012.index, profile_2012["mean"], "o-", markersize=3, label="June 2012")
ax.plot(profile_december.index, profile_december["mean"], "s--", markersize=3, label="December 2012")
ax.set(xlabel="Clock hour", ylabel="Mean rental count", xticks=range(0, 24, 3), ylim=(0, 750))
ax.legend(frameon=False)
plt.show()

In [ ]:
print("Rows in December:", len(december_2012))
print("Dates in December:", december_2012["date"].nunique())
profile_december.loc[profile_december["recorded_hours"] < 31].round(2)

In [ ]:
checks = pd.Series({
    "Rows accounted for": profile_2012["recorded_hours"].sum() == len(june_2012),
    "Rentals accounted for": profile_2012["total"].sum() == june_2012["cnt"].sum(),
    "Eight o'clock matches": profile_2012.loc[8, "mean"] == eight_in_june.mean(),
})
checks

12 月の 3 時台と 4 時台は各 30 行，ほかの時間帯は 31 行です。同じ平均の計算でも，分母となる行数が違います。記録がない時間帯を貸出 0 件として追加したわけではありません。

照合セルの各 `True` が，行数，貸出合計，8 時台の平均のどれを確かめているかを説明してください。同じ処理を繰り返せることに加えて，計算した値が何を表すかを確かめる作業です。実習では，この一連の処理を新しい実行状態から再現します。

## Jupyter 実習（約 20 分）

ここからは登録・未登録のどちらかを選び，2012 年の 6 月と 12 月で利用の多い時間帯がどう違うかを調べます。期間と区分を明記して，図の値を後から確かめられる報告にします。

関数の返り値の役割を自分で確かめるため，課題セルで集計関数を定義し直します。講義で計算した表をそのまま使わず，選んだ区分を二つの月の入力に適用してください。

In [ ]:
june = hourly.loc[(hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 6)]
december = hourly.loc[(hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 12)]
print("June rows:", len(june), "December rows:", len(december))

## 1. 比べる区分を選ぶ（約 5 分）

本文では，まず全体の貸出平均を比べました。2012 年 6 月の 8 時台は 508.4 件，17 時台は 680.7 件でした。12 月ではそれぞれ約 359.81 件と 375.97 件で，朝夕の平均件数の差が小さくなりました。今回は，選んだ利用区分でも同じような違いがあるかを調べます。

今回は `registered`（登録）か `casual`（未登録）の一方を，結果を見る前に選びます。朝・昼・夕方のどの利用を知りたいかと，その区分を選んだ理由を一文で書いてください。どちらの区分も，利用目的を直接表す列ではありません。

- 選ぶ区分と理由：
- 6 月と 12 月で朝・昼・夕方のどの違いを調べたいか：

次の二つのセルで，**合計二行**を完成します。関数は入力表を `hr` で集計済みです。まず `return None` を，集計表を返す行へ書き換えます。次に `selected_column = None` を，選んだ列名の文字列へ書き換えます。未記入のままでは後続セルに課題未完と表示されます。

In [ ]:
def hourly_summary(records, value_column):
    grouped = records.groupby("hr")[value_column]
    summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
    return None  # 課題：集計表 summary を返す行に替える。

In [ ]:
selected_column = None  # 課題："registered" または "casual" を記入する。

## 2. 二つの月を同じ関数で集計する（約 9 分）

次のセルは，選んだ区分について `hourly_summary(june, selected_column)` と `hourly_summary(december, selected_column)` を呼びます。対象月を表す入力表だけが変わり，選んだ列と集計方法は共通です。呼び出しの引数が関数のどのパラメーターに渡るかを確認してから実行してください。

返された表の 3・4・8・13・17 時台を並べ，続いて折れ線を描きます。平均の高さを比べるときには，`recorded_hours` の行数も一緒に読みます。6 月は 30 日分ですが，12 月には 30 日分の時間帯と 31 日分の時間帯があるので，どの日を平均した結果かを説明に含めます。

In [ ]:
june_profile = None
december_profile = None
if selected_column not in ["registered", "casual"]:
    print("課題未完：selected_column に選んだ列名を文字列で指定してください。")
else:
    june_profile = hourly_summary(june, selected_column)
    december_profile = hourly_summary(december, selected_column)
    if june_profile is None or december_profile is None:
        print("課題未完：関数の return で集計表を返してください。")
    else:
        comparison = pd.concat(
            {"June": june_profile, "December": december_profile}, names=["period"]
        )
        display(comparison.loc[(slice(None), [3, 4, 8, 13, 17]), :].round(2))

In [ ]:
if june_profile is None or december_profile is None:
    print("課題未完：返り値の表ができたら，このセルを再実行してください。")
else:
    titles = {"registered": "Registered rentals", "casual": "Casual rentals"}
    fig, ax = plt.subplots(layout="constrained", figsize=(7, 4))
    ax.plot(june_profile.index, june_profile["mean"], "o-", markersize=3, label="June 2012")
    ax.plot(december_profile.index, december_profile["mean"], "s--", markersize=3, label="December 2012")
    upper = max(june_profile["mean"].max(), december_profile["mean"].max()) * 1.1
    ax.set(xlabel="Clock hour", ylabel="Mean rental count", title=titles[selected_column],
           xticks=range(0, 24, 3), ylim=(0, upper))
    ax.legend(frameon=False)
    plt.show()
    print("Highest mean in June:", june_profile["mean"].idxmax(), "h")
    print("Highest mean in December:", december_profile["mean"].idxmax(), "h")

図で利用の多い時間帯を一つ選び，返り値の対応する行を見ます。その時間帯の合計・行数・平均を記録し，合計を行数で割ると図の点の高さになるかを確かめてください。6 月と 12 月で，利用の多い時間帯や件数がどう違うかも一文にします。

12 月の 3 時台と 4 時台は各 30 行の平均です。記録がない日を 0 件として含めた平均ではありません。対象にした時間帯の行数を使い，「記録がある日についての平均」だと分かる説明にしてください。

- 図で選んだ時間帯と，合計・分母・平均：
- 6 月と 12 月の違い：
- この図だけではまだ分からないこと：

## 3. 最初から再実行して結果を照合する（約 6 分）

別の人が同じ図を作れるかを確かめるため，ここまでのセルを保存し，カーネル（Colab ではランタイム）を再起動して，上から全セルを実行します。記入したコードは保存されますが，前の実行で作った変数や関数は消え，ノートにある処理から作り直されます。

次のセルでは，時刻別の合計を全部足すと入力表の貸出合計に戻るか，行数の合計が元の行数に戻るか，8 時台の平均が元の記録から直接求めた平均と一致するかを確かめます。再び図が出ることと，その図の値が意図した記録から計算されていることを，両方確認してください。

In [ ]:
if june_profile is None or december_profile is None:
    print("課題未完：二つの返り値ができてから照合します。")
else:
    checks = []
    for label, records, profile in [
        ("June", june, june_profile), ("December", december, december_profile)
    ]:
        eight = records.loc[records["hr"] == 8, selected_column]
        checks.append({
            "period": label,
            "rows_match": profile["recorded_hours"].sum() == len(records),
            "totals_match": profile["total"].sum() == records[selected_column].sum(),
            "eight_matches": profile.loc[8, "mean"] == eight.mean(),
        })
    display(pd.DataFrame(checks))

次の項目が伝わるよう，図と短表に添える 3〜4 文の報告を書きます。

- 入力データの版，期間，選んだ区分。
- 同じ時間帯の貸出件数を平均したことと，二つの月で利用の多い時間帯や平均件数がどう違ったか。
- 比べた時間帯に何日分の記録があるか。利用目的や，月によって件数が違う理由について，この比較だけでは分からないこと。
- 最初から再実行できたか，どの出力を元の記録と照合したか。

（ここに報告を記入）

今回の分析は乱数を使っていません。第 10 章の並べ替えのように乱数を使う分析を再現する場合には，追加で何を記録するか一つ挙げてください。

<details>
<summary>記入方法と，結果を確かめる手掛かり</summary>

まず講義の「同じ集計を使える関数にする」で，関数の最後の行を確認してください。集計表を後の計算へ渡すには，`return summary` と書きます。列名の指定は，登録区分を選ぶなら `selected_column = "registered"` とします。未登録区分を選んだ場合は，対応する列名へ替えてください。

実行後は，自分が選んだ区分の図で，6 月と 12 月の利用が多い時間帯を探します。その時間帯の行を返り値の表で選び，平均件数を確かめてください。最大となる時間帯だけでなく，その前後にも利用の多い時間が続くかを見ると，担当者への説明を具体的にできます。

どちらの区分でも，12 月の 3 時台と 4 時台は各 30 行から求めた平均です。再実行と照合ができたことから，記録がない時間帯の利用まで 0 件だったとは判断できません。登録・未登録の区分も，利用目的そのものを示すものではありません。

</details>

## 出典

Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository，CC BY 4.0。 [配布元](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) ・ [データ DOI](https://doi.org/10.24432/C5W894)

本書の `bike-sharing-hourly.csv` は UCI 公式 ZIP の `hour.csv` を改名した集計版です。列名・値・行順は同じです。現在の事業者の個票ログとは版が異なり，混ぜて使いません。日付・時計時刻・欠落時刻の性格は Book の出典説明と第 11 章を参照してください。